# DosePy：全剂量预测（Full Dose Prediction）全流程教程（Anatomy-only 7 通道）

本 notebook 是 PortPy total-dose 代码的 **anatomy-only 7 通道版本**：输入病人的 CT、OAR mask 与 PTV，不含 BEAM 通道；预测整个疗程的三维剂量分布（Gy），并完成评估与可视化。

本教程按下面的顺序展开，每一步都对应 `DosePy/dosepy` 中的一个模块：

```text
DICOM + metadata
      │  dosepy.preprocess.data_preprocess
      ▼
128³ npz（CT / OAR / PTV / DOSE / BODY）
      │  dosepy.preprocess.split_preprocessed_dataset
      ▼
train / val / test
      │  dosepy.train.train
      ▼
checkpoints（latest / best_mae / <epoch>）
      │  dosepy.test.test 或 dosepy.preprocess.predict_using_model
      ▼
CT 网格上的预测剂量（Gy）
      │  dosepy.evaluation + dosepy.visualization
      ▼
指标 CSV / DVH / 切片图 / scorecard
```

> **说明**：本 notebook 的所有 code cell 都没有执行，也不包含运行输出。运行前需要准备 PortPy 数据、安装依赖、最好有 GPU。没有 GPU 时把 `gpu_ids` 设为 `[]` 可以跑通小规模 smoke test，但训练会非常慢。

## 目录

0. 环境与路径
1. 数据预处理
2. 数据划分（train / val / test）
3. 检查预处理结果
4. 选择模型
5. 定义参数
6. 训练
7. 测试
8. 单病例推理
9. 评估（体素指标、DVH、scorecard）
10. 可视化
11. 下游：用预测剂量做通量优化
12. 自定义与扩展
13. 常见问题（FAQ）


## 0. 环境与路径

### 0.1 依赖

- `portpy`：数据访问（`DataExplorer` / `AnatomyOnlyDataExplorer`、`CT`、`Structures`）；评估/下游还需要 `Beams`、`InfluenceMatrix`、`Plan`、`Optimization`
- `torch`：训练/推理
- `SimpleITK`、`pydicom`、`scikit-image`：DICOM 读取与几何重采样
- `pandas`、`matplotlib`：评估与可视化
- 可选：`nnunet_mednext`（`mednext*` 模型）、`Mosek`（下游优化）

安装：

```bash
cd DosePy
pip install -r requirements.txt
pip install -e .
# 可选
pip install git+https://github.com/MIC-DKFZ/MedNeXt.git
pip install Mosek
```

### 0.2 PortPy 数据

`DATA_DIR` 应指向 PortPy 的数据根目录，即包含 `Lung_Patient_*` / `Prostate_Patient_*` 文件夹的目录。每个病人至少需要：

- `CT_MetaData.json`
- `OptimizationVoxels_MetaData.json`（calculation box）
- `StructureSet_MetaData.json`
- `DicomFiles/rt_dose_echo_<technique>.dcm`（训练/验证需要；推理不需要）
- `PlannerBeams.json` / `Beams/` / influence matrix（仅评估、scorecard 和下游优化需要；预测本身不需要）

如果还没有数据，可以先下载一个病人：

```python
import portpy.photon as pp
pp.download_portpy_data(["Lung_Patient_2"], out="./portpy_data", beam_mode="planner")
# 只做 prediction 可用 beam_mode="none"；本 notebook 后面有 scorecard/下游优化，所以保留 planner beams。
```


In [ ]:
# 把 DosePy 根目录加入 sys.path。
# 兼容三种启动目录：DosePy/、DosePy/notebooks/、PortPy/（仓库根目录）。
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
DOSEPY_ROOT = None
for candidate in [CWD, *CWD.parents, CWD / "DosePy"]:
    if (candidate / "dosepy" / "__init__.py").exists():
        DOSEPY_ROOT = candidate
        break

if DOSEPY_ROOT is None:
    DOSEPY_ROOT = CWD
    print("警告：没有找到 dosepy/__init__.py；如果你已经 pip install -e DosePy，可以直接 import。")

if str(DOSEPY_ROOT) not in sys.path:
    sys.path.insert(0, str(DOSEPY_ROOT))

print("DosePy root:", DOSEPY_ROOT)


In [ ]:
# 常用库与 DosePy API
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import portpy.photon as pp

import dosepy
from dosepy.configs import scorecard_path
from dosepy.downstream import optimize_fluence_to_prediction
from dosepy.evaluation import (
    dose_metrics,
    dose_metrics_by_structure,
    dvh_metrics,
    dvh_table,
    get_plan_score,
    load_scorecard_json,
)
from dosepy.options import TestOptions, TrainOptions, build_opt
from dosepy.preprocess import (
    AnatomyOnlyDataExplorer,
    check_split_integrity,
    data_preprocess,
    get_benchmark_split,
    get_channel_names,
    get_dvh_structure_names,
    get_site_profile,
    predict_using_model,
    split_preprocessed_dataset,
)
from dosepy.test import test
from dosepy.train import train
from dosepy.visualization import (
    load_training_history,
    plot_dvh_comparison,
    plot_patient_metric_distribution,
    plot_plan_2d_slice,
    plot_plan_dvh,
    plot_preprocessed_sample,
    plot_slice_comparison,
    plot_structure_metric_bars,
    plot_training_history,
    save_evaluation_report,
)

print("dosepy", dosepy.__version__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# ---------------------------------------------------------------------------
# 路径配置：按你的实际环境修改，或设置同名环境变量
# ---------------------------------------------------------------------------
DATA_DIR = os.environ.get("PORTPY_DATA_DIR", "/path/to/PortPy/data")
WORK_DIR = os.environ.get("DOSEPY_WORK_DIR", str(DOSEPY_ROOT / "workspace"))

PREPROCESS_DIR = os.path.join(WORK_DIR, "preprocessed_lung")
CHECKPOINTS_DIR = os.path.join(WORK_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
INFER_INPUT_DIR = os.path.join(WORK_DIR, "infer_input")
REPORT_DIR = os.path.join(WORK_DIR, "reports")

SITE = "lung"                       # "lung" 或 "prostate"
PROTOCOL = "Lung_2Gy_30Fx"          # lung: Lung_2Gy_30Fx；prostate: Prostate_26Fx
TECHNIQUE = "imrt"                  # 对应 rt_dose_echo_<technique>.dcm
EXPERIMENT = "dosepy_lung_unet128"  # checkpoint / results 下的实验名
PATIENT_ID = "Lung_Patient_94"      # 单病例推理示例（属于 val 划分）

# 有 GPU 用 [0]；没有 GPU 用 []（很慢，只建议 smoke test）
GPU_IDS = [0] if torch.cuda.is_available() else []

for d in (WORK_DIR, CHECKPOINTS_DIR, RESULTS_DIR, INFER_INPUT_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

print("DATA_DIR      :", DATA_DIR)
print("PREPROCESS_DIR:", PREPROCESS_DIR)
print("GPU_IDS       :", GPU_IDS)


> 如果 `DATA_DIR` 仍是 `/path/to/PortPy/data`，后面的 cell 会失败。请先设置环境变量：
>
> ```bash
> export PORTPY_DATA_DIR=/your/path/to/portpy_data/data
> export DOSEPY_WORK_DIR=/your/path/to/work
> ```
>
> 或者在 Jupyter 中修改上面的常量。

## 1. 数据预处理

### 1.1 预处理做了什么

`data_preprocess` 对每个病人执行：

1. 读取 PortPy metadata，拿到 calculation box（`opt_voxels.cal_box_xyz_start/end`）；
2. 把 CT、OAR 标签、PTV、planner dose 裁剪到 calc box；
3. 重采样到 `128 × 128 × 128`：
   - CT / dose：线性插值（`order=1`）
   - OAR / PTV / BODY：最近邻（`order=0`）
4. 把 OAR 栅格化为标签图，PTV 单独保存为二值 mask；
5. planner dose 裁剪到 `[0, 1.2 × 处方剂量]`；
6. CT HU 裁剪到 `[-1000, 3071]`，再线性缩放到 `[0, 1]`；
7. 保存为 `<Patient>.npz`。

整个流程**不需要 beams、`Beams/` 元数据或 influence matrix**。

生成的文件键：

| key | shape | dtype | 含义 |
|---|---|---|---|
| `CT` | `(128,128,128)` | float32 | 归一化 CT |
| `DOSE` | `(128,128,128)` | float32 | planner dose（仅训练/验证） |
| `OAR` | `(128,128,128)` | uint8 | OAR 标签 0–4 |
| `PTV` | `(128,128,128)` | uint8 | PTV 0/1 |
| `BODY` | `(128,128,128)` | uint8 | BODY 0/1 |
| `PRESCRIPTION_GY` | scalar | float32 | 处方剂量 |

### 1.2 先确认 site 配置

> 本版本是 anatomy-only：输入只有 CT + OAR + PTV，没有 BEAM。PortPy 官方 8 通道 checkpoint 不能加载，需要重新训练。

DosePy 支持的 site 配置固定写在 `dosepy/preprocess/site_config.py`。**channel 顺序不能改**，否则 DosePy 7 通道 checkpoint 无法加载。


In [ ]:
profile = get_site_profile(SITE)
print("site              :", profile.site)
print("structures        :", profile.disease_struct_names)
print("labels            :", profile.labels)
print("prescription (Gy) :", profile.prescription_gy)
print("default protocol  :", profile.default_protocol)
print("dvh structures    :", profile.dvh_structures)
print("input channels    :", get_channel_names(SITE))


### 1.3 批量预处理

下面的 cell 会遍历 `DATA_DIR` 中所有名字包含 `site` 的病人。运行时间取决于病人数量（每个病人都要算 influence matrix），第一次跑会比较久。

- 已存在同名 `.npz` 时默认跳过，方便断点续跑；需要重跑加 `overwrite=True`。
- 没有 `rt_dose_echo_<technique>.dcm`、没有 PTV、没有 BODY 的病人会被跳过并打印原因。
- `protocol_name` 决定处方剂量；`None` 时使用 site 默认处方（lung 60 Gy，prostate 70.2 Gy）。

> **Smoke test 提示**：完整预处理整个 cohort 需要较长时间。如果只想验证流程，可以给 `patient_ids` 传 2–3 个病人（最好 train/val/test 各一个），但训练结果没有统计意义。


In [ ]:
# 默认处理 DATA_DIR 中所有 lung 病人。
# 若只想 smoke test，可加 patient_ids=["Lung_Patient_2", "Lung_Patient_92", "Lung_Patient_82"]，
# 这样 train/val/test 各至少有一个文件，但训练结果没有统计意义。
processed = data_preprocess(
    in_dir=DATA_DIR,
    out_dir=PREPROCESS_DIR,
    site=SITE,
    protocol_name=PROTOCOL,
    technique_name=TECHNIQUE,
    overwrite=False,
    verbose=True,
)
print("processed:", processed)


## 2. 数据划分（train / val / test）

PortPy.AI 使用病人级别的冻结划分。**Task 2（全剂量）和 Task 1（beamlet）的 lung validation/test 是交换的**，不能混用。DosePy 的 `get_benchmark_split(site)` 默认读取 `total_dose_prediction`，这是正确划分。

`data_preprocess` 输出的是扁平目录，`split_preprocessed_dataset` 会复制（默认）到：

```text
preprocessed_lung/
├── train/Lung_Patient_2.npz ...
├── val/Lung_Patient_91.npz ...
├── test/Lung_Patient_81.npz ...
└── split_manifest.json
```

> 如果磁盘紧张，可以用 `mode="move"` 或 `mode="symlink"`。

In [ ]:
splits = get_benchmark_split(SITE)
print({k: len(v) for k, v in splits.items()})
print("train (前 5 个):", splits["train"][:5])
print("val   (前 5 个):", splits["val"][:5])
print("test  (前 5 个):", splits["test"][:5])


In [ ]:
placed = split_preprocessed_dataset(
    out_dir=PREPROCESS_DIR,
    site=SITE,
    task="total_dose_prediction",
    mode="copy",       # "copy" | "move" | "symlink"
    overwrite=False,
    verbose=True,
)
print(check_split_integrity(PREPROCESS_DIR))


## 3. 检查预处理结果

在训练前一定要抽查一个 `.npz`，确认：

- 数组形状都是 `(128,128,128)`；
- `OAR` 标签只包含 0–4；
- `PTV`、`BODY` 是 0/1；
- `DOSE` 最大值不超过 `1.2 × 处方剂量`；
- CT 在 `[0,1]`。

`plot_preprocessed_sample` 会画出 CT / OAR / PTV / BODY / DOSE 面板（推理 npz 没有 DOSE 时少一个）。


In [ ]:
train_dir = Path(PREPROCESS_DIR) / "train"
sample_path = train_dir / (PATIENT_ID + ".npz")
if not sample_path.exists():
    candidates = sorted(train_dir.glob("*.npz"))
    sample_path = candidates[0] if candidates else None

print("sample:", sample_path)
if sample_path is not None and sample_path.exists():
    with np.load(sample_path, allow_pickle=True) as d:
        for key in d.files:
            arr = d[key]
            if arr.dtype == object:
                print(f"{key:>16}: shape={arr.shape}, dtype=object (旧版 PortPy 文件)")
                continue
            print(f"{key:>16}: shape={arr.shape}, dtype={arr.dtype}, "
                  f"min={np.min(arr):.4f}, max={np.max(arr):.4f}")
else:
    print("没有找到预处理文件，请先运行第 1 节的 data_preprocess。")


In [ ]:
if sample_path is not None and sample_path.exists():
    plot_preprocessed_sample(sample_path, site=SITE)
    plt.show()
else:
    print("跳过：没有可用的 npz。")


### 3.1 7 通道布局（必须理解）

`transform_3d_data` 实际构造的输入是：

```text
[CT, OAR-label-1, OAR-label-2, OAR-label-3, OAR-label-4, OAR-label-5(空), PTV]
```

- `OAR` 的标签 1–4 来自 `site_config.py`；标签 5 没有使用，但为了固定 7 通道布局而保留；
- PTV 单独放在最后一个通道；
- 因此 `real_CT[:, 1:, ...]` 是 6 个结构通道（4 个 OAR + 1 个空 + PTV），`real_CT[:, -1:, ...]` 是 PTV；
- **没有 BEAM 通道**，所以模型输入不需要 beams、`Beams/` 元数据或 influence matrix。

> 如果你要加载 DosePy 7 通道 checkpoint，**不要改动这个布局**，也不要改 CT 的归一化方式。PortPy 官方 8 通道 checkpoint 不兼容。


## 4. 选择模型

### 4.1 模型对比

| netG | 结构 | 参数量（约） | 依赖 | 适用场景 |
|---|---|---|---|---|
| `unet_128` | 3D U-Net，7 次下采样 | ~16 M（ngf=64） | 无 | 最小可运行基线，PortPy 历史默认 |
| `unet_256` | 3D U-Net，8 次下采样 | 更大 | 无 | 数据量较大、想增加容量 |
| `mednext` | MedNeXt-B | ~10.5 M | `nnunet_mednext` | GDP-HMM Top-1 风格基线 |
| `mednext_attn` | MedNeXt + attention-gated skip | ~10.5 M+ | `nnunet_mednext` | Top-2，注意力增强 |
| `mednext_duo` | 共享 encoder + 每 site decoder | 取决于 `mednext_n_channels` | `nnunet_mednext` | 多 site 联合训练（lung + prostate） |

**选择建议**：

- 第一次跑通流程：`unet_128` + `mae`；
- 追求更好精度、有 GPU 显存：`mednext` + `mednext_hybrid`；
- 同时训练 lung 和 prostate：`mednext_duo`，并把 `mednext_sites` 设为 `["lung", "prostate"]`；
- 不建议一开始就上 `unet_256` 或 `mednext_duo`，显存占用和调参成本都更高。

### 4.2 损失函数

| dose_loss | 组成 | 说明 |
|---|---|---|
| `mae` | L1 | 最稳定，PortPy 原始默认 |
| `mae_moment` | L1 + ROI 剂量矩 | 额外约束每个结构的剂量分布矩 |
| `mednext_hybrid` | weighted L1 + body-masked weighted L1 + 剂量矩 + dual-gradient | MedNeXt 推荐；需要 `BODY` |

`mednext_hybrid` 的权重通过 `lambda_wL1`、`lambda_masked_wL1`、`lambda_moment` 控制。


In [ ]:
# 先选一个稳妥的基线
NET_G = "unet_128"
DOSE_LOSS = "mae"

# 可选（需要先安装 nnunet_mednext）：
# NET_G = "mednext"
# DOSE_LOSS = "mednext_hybrid"

print("netG      :", NET_G)
print("dose_loss :", DOSE_LOSS)


In [ ]:
# 可选：打印模型参数量，确认配置
from dosepy.models import networks3d

if NET_G in ("unet_128", "unet_256"):
    _net = networks3d.define_G(8, 1, 64, NET_G, norm="batch", gpu_ids=[])
    n_params = sum(p.numel() for p in _net.parameters()) / 1e6
    print(f"{NET_G}: {n_params:.2f} M parameters")
else:
    print(f"{NET_G}: 需要 nnunet_mednext，跳过参数量统计（避免实例化失败）")


## 5. 定义参数

DosePy 的参数可以用一个 Python dict 表示，既可以直接传给 `train()` / `test()`，也可以用 `build_opt` 先检查最终 namespace。

### 5.1 训练参数

下面是最小可用配置。完整参数列表见 `dosepy/options/train_options.py`。

| 参数 | 说明 |
|---|---|
| `dataroot` | 含 `train/val/test` 的预处理根目录 |
| `checkpoints_dir` / `name` | checkpoint 根目录 / 实验名 |
| `model` | `doseprediction3d`（训练/验证/测试） |
| `dataset_mode` | `dosepred3d`（有 GT） |
| `netG` / `dose_loss` | 见第 4 节 |
| `input_nc` / `output_nc` | 7 / 1（anatomy-only） |
| `ngf` | U-Net 基础通道数，默认 64 |
| `norm` | U-Net 归一化：`batch` / `instance` / `none` |
| `batch_size` | 3D 数据通常只能 1–2 |
| `num_threads` | DataLoader 线程数 |
| `lr` / `lr_policy` | 学习率 / `plateau`、`linear`、`step`、`cosine` |
| `n_epochs` / `n_epochs_decay` | 训练轮数 |
| `lambda_L1` / `lambda_wL1` / `lambda_masked_wL1` / `lambda_moment` | 各损失权重 |
| `augment` | 随机 3D 翻转/旋转/平移 |
| `val_phase` | 验证集目录名，默认 `val` |
| `gpu_ids` | `[0]` 用 GPU；`[]` 用 CPU |
| `save_epoch_freq` | 每隔多少 epoch 保存数字 checkpoint |
| `display_id` | `-1` 关闭 visdom（推荐） |


In [ ]:
train_options = {
    "dataroot": PREPROCESS_DIR,
    "checkpoints_dir": CHECKPOINTS_DIR,
    "name": EXPERIMENT,

    # 模型
    "model": "doseprediction3d",
    "dataset_mode": "dosepred3d",
    "netG": NET_G,
    "dose_loss": DOSE_LOSS,
    "input_nc": 7,
    "output_nc": 1,
    "ngf": 64,
    "norm": "batch",

    # 优化
    "batch_size": 1,
    "num_threads": 2,
    "lr": 1e-4,
    "lr_policy": "plateau",
    "n_epochs": 100,
    "n_epochs_decay": 0,
    "lambda_L1": 1.0,
    "lambda_wL1": 1.0,
    "lambda_masked_wL1": 1.0,
    "lambda_moment": 0.1,

    # 数据 / 日志
    "augment": False,
    "val_phase": "val",
    "display_id": -1,
    "no_html": True,          # 不保存中间 nrrd/HTML，节省磁盘
    "display_freq": 10,
    "print_freq": 1,
    "save_epoch_freq": 10,
    "save_latest_freq": 500,
    "gpu_ids": GPU_IDS,
}

# 第一次跑建议先把 n_epochs 改成 2，确认流程通，再放开
# train_options["n_epochs"] = 2


### 5.2 用 `build_opt` 检查最终参数

`build_opt` 会先按默认值构造 parser，再应用 dict。**未知参数名会直接报错**，可以避免拼写错误。

这里把 `gpu_ids` 临时改成 `[]`，这样没有 GPU 也能检查参数（不会调用 `torch.cuda.set_device`）。


In [ ]:
preview = build_opt(TrainOptions, {**train_options, "gpu_ids": []})

keys = [
    "dataroot", "name", "model", "dataset_mode", "netG", "dose_loss",
    "input_nc", "output_nc", "batch_size", "lr", "lr_policy",
    "n_epochs", "val_phase", "augment", "save_epoch_freq",
]
for key in keys:
    print(f"{key:>18}: {getattr(preview, key)}")


### 5.3 测试参数

测试有两种模式：

1. **有 GT 的评估**：`model="doseprediction3d"`、`dataset_mode="dosepred3d"`，会计算指标并写 `patient_metrics.csv`；
2. **纯推理**：`model="test"`、`dataset_mode="single"`，只保存预测剂量，不需要 GT。

`epoch` 可以是 `latest`、`best_mae` 或数字 epoch（例如 `"50"`）。


In [ ]:
test_options = {
    "dataroot": PREPROCESS_DIR,
    "checkpoints_dir": CHECKPOINTS_DIR,
    "results_dir": RESULTS_DIR,
    "name": EXPERIMENT,

    "model": "doseprediction3d",
    "dataset_mode": "dosepred3d",
    "netG": NET_G,
    "dose_loss": DOSE_LOSS,
    "input_nc": 7,
    "output_nc": 1,
    "norm": "batch",

    "phase": "test",
    "epoch": "best_mae",
    "eval": True,
    "gpu_ids": GPU_IDS,
}

# 纯推理模式（predict_using_model 内部会自动使用）：
# {"model": "test", "dataset_mode": "single", ...}


## 6. 训练

`train(train_options)` 会：

1. 创建 train / val DataLoader；
2. 每个 epoch 训练，然后在 val 上计算 loss、challenge-style MAE、R²、Pearson、nRMSE；
3. 用 val MAE 驱动 `ReduceLROnPlateau`；
4. 保存 `latest`、数字 epoch、`best_mae`；
5. 把指标写入 `<checkpoints_dir>/<name>/training_history.csv`；
6. **返回 history dict**，可以直接画图。

> 这一 cell 需要 GPU/较长时间。没有条件运行时不要执行；参数和代码仍然可以阅读。


In [ ]:
history = train(train_options)


训练输出目录：

```text
checkpoints/<name>/
├── latest_net_G.pth
├── best_mae_net_G.pth
├── 10_net_G.pth
├── ...
├── training_history.csv
├── console_output.txt
├── loss_log.txt
└── web/                  # 如果 no_html=False，保存中间 nrrd/html
```

`best_mae_net_G.pth` 是按验证集 challenge-style MAE 选出的最佳模型，测试和推理通常用它。


In [ ]:
# 如果本次 session 里训练过，直接用返回值；
# 否则从 CSV 读取（方便第二天继续分析）
history_csv = os.path.join(CHECKPOINTS_DIR, EXPERIMENT, "training_history.csv")

if "history" in globals() and history:
    plot_training_history(history)
    plt.show()
elif os.path.exists(history_csv):
    plot_training_history(load_training_history(history_csv))
    plt.show()
else:
    print("还没有训练历史：", history_csv)


### 6.1 命令行等价写法

```bash
python -m dosepy.train \
  --dataroot /path/to/preprocessed_lung \
  --checkpoints_dir /path/to/checkpoints \
  --name dosepy_lung_unet128 \
  --model doseprediction3d \
  --dataset_mode dosepred3d \
  --netG unet_128 \
  --dose_loss mae \
  --input_nc 7 --output_nc 1 \
  --batch_size 1 --gpu_ids 0 \
  --n_epochs 100 --lr 1e-4 --lr_policy plateau
```

继续训练：

```python
train_options["continue_train"] = True
train_options["epoch"] = "latest"   # BaseModel.setup 会加载 latest
```


## 7. 测试

`test(test_options)` 遍历 `dataroot/test`：

- 保存每个病人的预测剂量到 `results/<name>/test_<epoch>/npz_images/<Patient>_CT2DOSE.nrrd`；
- 计算 loss、MAE、RMSE、R²、Pearson、nRMSE；
- 写 `patient_metrics.csv`；
- 生成 `index.html` HTML 索引（剂量以 `.nrrd` 保存，PNG 预览用 DosePy 的绘图函数）。


In [ ]:
summary = test(test_options)


In [ ]:
print("评估病例数 :", summary["n_eval"])
print("平均 loss  :", summary["avg_loss"])
print("平均 MAE   :", summary["avg_mae"])
print("结果目录   :", summary["web_dir"])

metrics_csv = os.path.join(summary["web_dir"], "patient_metrics.csv")
if os.path.exists(metrics_csv):
    df = pd.read_csv(metrics_csv)
    print(df.head(10).to_string(index=False))
    plot_patient_metric_distribution(metrics_csv, metric="mae")
    plt.show()
else:
    print("没有 patient_metrics.csv（可能没有 GT，或模型是 test 模式）。")


## 8. 单病例推理

`predict_using_model` 会完成：预处理 → 加载模型 → 推理 → 重采样回 CT 网格 → 返回 `(D,H,W)` 剂量。

注意：

- 它使用与训练**完全相同**的 anatomy-only 预处理，包括 CT 归一化、OAR/PTV mask 和 calc-box 裁剪；
- 推理目录是 `<out_dir>/infer/<patient_id>/`，不会污染其他病人的 npz；
- `epoch` 可以是 `best_mae` / `latest` / 数字；
- `netG` 必须与训练一致，否则 checkpoint 结构不匹配；
- 预测本身**不建 BEAM**，也不加载 `Beams` / `InfluenceMatrix`。


In [ ]:
# 先加载预测需要的 PortPy 对象（CT / structures / prescription）
# 用 AnatomyOnlyDataExplorer：没有 Beams/ 也能读取 CT/structures
# 注意：预测本身不需要 beams 或 influence matrix
data = AnatomyOnlyDataExplorer(data_dir=DATA_DIR)
data.patient_id = PATIENT_ID

ct = pp.CT(data)
structs = pp.Structures(data)
clinical_criteria = pp.ClinicalCriteria(data, protocol_name=PROTOCOL)
RX = clinical_criteria.get_prescription()

print("patient      :", PATIENT_ID)
print("CT shape     :", ct.ct_dict["ct_hu_3d"][0].shape)
print("prescription :", RX, "Gy")
print("num fractions:", clinical_criteria.get_num_of_fractions())


In [ ]:
pred_dose_3d = predict_using_model(
    patient_id=PATIENT_ID,
    in_dir=DATA_DIR,
    out_dir=INFER_INPUT_DIR,
    model_name=EXPERIMENT,
    checkpoints_dir=CHECKPOINTS_DIR,
    results_dir=RESULTS_DIR,
    netG=NET_G,
    site=SITE,
    protocol_name=PROTOCOL,
    epoch="best_mae",
    save_nrrd=True,
)

print("pred shape:", pred_dose_3d.shape)
print("pred range: [{:.2f}, {:.2f}] Gy".format(float(pred_dose_3d.min()), float(pred_dose_3d.max())))


### 8.1 读取 planner dose 作为 GT

PortPy 的 planner dose 存在 `DicomFiles/rt_dose_echo_<technique>.dcm`，用 `pp.convert_dose_rt_dicom_to_portpy` 转换到 CT 网格。


In [ ]:
gt_dose_3d = pp.convert_dose_rt_dicom_to_portpy(
    ct=ct,
    dose_file_name=os.path.join(
        DATA_DIR, PATIENT_ID, "DicomFiles", "rt_dose_echo_{}.dcm".format(TECHNIQUE)
    ),
)

print("GT shape:", gt_dose_3d.shape)
print("GT range: [{:.2f}, {:.2f}] Gy".format(float(gt_dose_3d.min()), float(gt_dose_3d.max())))


### 8.2 构造结构 mask

评估和可视化需要 `dict[str, bool mask]`。只保留该病人实际存在的结构。


In [ ]:
name_map = {s.upper(): s for s in structs.get_structures()}
structure_masks = {}
for name in get_dvh_structure_names(SITE):
    if name.upper() not in name_map:
        print("skip (missing):", name)
        continue
    actual = name_map[name.upper()]
    mask = structs.get_structure_mask_3d(actual) > 0
    if not mask.any():
        print("skip (empty):", name)
        continue
    structure_masks[name] = mask

print("structures:", list(structure_masks))
for name, mask in structure_masks.items():
    print(f"{name:>16}: {mask.sum()} voxels")


### 8.3 批量推理：复用 `InferenceRunner`

对多个病人推理时，`predict_using_model` 每个病人都会重新加载权重。可以先构造 `InferenceRunner`，权重只加载一次：

```python
from dosepy.infer import InferenceRunner

runner = InferenceRunner({
    "checkpoints_dir": CHECKPOINTS_DIR,
    "results_dir": RESULTS_DIR,
    "name": EXPERIMENT,
    "netG": NET_G,
    "epoch": "best_mae",
    "eval": True,
    "gpu_ids": GPU_IDS,
})

for pid in splits["test"][:5]:
    # 先用与 predict_using_model 相同的预处理写出 <infer_dir>/<pid>.npz
    infer_dir = ...   # 见 dosepy/preprocess/predict.py 的 _prepare_inference_dir
    runner.run(infer_dir)
```

DosePy 没有为“批量预处理 + runner”再包一层，这是有意为之：预处理每个病人的位置/目录策略可能不同，保持 `predict_using_model` 简单、`InferenceRunner` 可复用即可。


## 9. 评估

评估分三层：

1. **体素级回归指标**（`dose_metrics`）：MAE、RMSE、R²、Pearson、nRMSE；
2. **分结构指标**（`dose_metrics_by_structure`）：每个结构的 MAE/RMSE 等；
3. **临床指标**：
   - DVH 指标（`dvh_metrics` / `dvh_table`）：D95、Dmean、V20Gy 等；
   - scorecard（`get_plan_score`）：PortPy 的评分卡，逐条 criterion 打分后求和。

> 体素指标回答“预测有多准”，scorecard 回答“这个剂量作为计划有多好”。两者不能互相替代。


### 9.1 整体指标

用 BODY 作为 mask（CT > -500 HU 是一个简单可靠的体表近似）。也可以直接用 `gt_dose_3d > 0` 或 `structs.get_structure_mask_3d("BODY") > 0`。


In [ ]:
ct_hu = ct.ct_dict["ct_hu_3d"][0]
body_mask = ct_hu > -500

overall = dose_metrics(
    pred_dose_3d,
    gt_dose_3d,
    mask=body_mask,
    prescription_gy=RX,
)

print(pd.Series(overall).to_string())


### 9.2 分结构指标

列含义：

- `MAE` / `RMSE`：Gy；
- `R2` / `Pearson`：越大越好；
- `nRMSE`：RMSE / GT 范围；
- `MAE(%Rx)` / `RMSE(%Rx)`：相对处方剂量的百分比；
- `MeanPred` / `MeanRef`：平均剂量；
- `MaxAbsErr`：最大绝对误差。


In [ ]:
per_structure = dose_metrics_by_structure(
    pred_dose_3d,
    gt_dose_3d,
    structure_masks,
    prescription_gy=RX,
)
print(per_structure.round(3).to_string(index=False))


### 9.3 DVH 指标

`dvh_table` 输出长表：每个结构 × 每个指标一行，包含预测值、参考值、差值。

- `D95`：95% 体积覆盖的剂量（Gy）；
- `V20Gy`：接受 ≥20 Gy 的体积百分比；
- `Dmean`：平均剂量。


In [ ]:
dvh_df = dvh_table(
    pred_dose_3d,
    gt_dose_3d,
    structure_masks,
    prescription_gy=RX,
    metrics=("D98", "D95", "D50", "D2", "Dmean", "V20Gy", "Dmax"),
)
print(dvh_df.round(2).to_string(index=False))


In [ ]:
# 单个结构的 DVH 指标（dvh_table 的底层函数）
for name in [n for n in ("PTV", "CORD", "ESOPHAGUS") if n in structure_masks]:
    pred_m = dvh_metrics(pred_dose_3d, structure_masks[name], prescription_gy=RX)
    ref_m = dvh_metrics(gt_dose_3d, structure_masks[name], prescription_gy=RX)
    print(name)
    print("  predicted: D95={:.2f} Gy, Dmean={:.2f} Gy, V20Gy={:.1f}%".format(
        pred_m["D95"], pred_m["Dmean"], pred_m["V20Gy"]))
    print("  reference: D95={:.2f} Gy, Dmean={:.2f} Gy, V20Gy={:.1f}%".format(
        ref_m["D95"], ref_m["Dmean"], ref_m["V20Gy"]))


### 9.4 scorecard（临床计划评分）

DosePy 自带从 PortPy 复制的 lung / prostate scorecard：

- `scorecard_path("lung")` → `SC_Lung(60Gy)_2022MAAS_ExampleV2.json`
- `scorecard_path("prostate")` → `SC_ProstateSBRT-ARTIA-V1.2(KBPModel).json`

scorecard 的一些结构名和 PortPy 计划里的名字不同，需要 `alias_map`：

- lung：`SPINAL_CORD → CORD`，`TOTAL LUNG - GTV → LUNGS_NOT_GTV`
- prostate：按实际结构名补充

**注意**：scorecard 的 `dose_1d` 必须是 optimization-voxel 网格上的 1D 剂量，用 `inf_matrix.dose_3d_to_1d` 转换。


In [ ]:
# 评估 / scorecard / 下游优化需要 beam 和 influence matrix；
# 预测本身（上一步的 predict_using_model）不需要它们。
beams = pp.Beams(data)
inf_matrix = pp.InfluenceMatrix(ct=ct, structs=structs, beams=beams)

my_plan = pp.Plan(
    ct=ct,
    structs=structs,
    beams=beams,
    inf_matrix=inf_matrix,
    clinical_criteria=clinical_criteria,
)

pred_dose_1d = inf_matrix.dose_3d_to_1d(pred_dose_3d)
gt_dose_1d = inf_matrix.dose_3d_to_1d(gt_dose_3d)

scorecard = load_scorecard_json(scorecard_path(SITE))
alias_map = {"SPINAL_CORD": "CORD", "TOTAL LUNG - GTV": "LUNGS_NOT_GTV"} if SITE == "lung" else {}

pred_score, pred_score_df = get_plan_score(
    my_plan,
    pred_dose_1d,
    scorecard,
    prescription_gy=RX,
    alias_map=alias_map,
    return_score_df=True,
)
gt_score, gt_score_df = get_plan_score(
    my_plan,
    gt_dose_1d,
    scorecard,
    prescription_gy=RX,
    alias_map=alias_map,
    return_score_df=True,
)

print("Predicted dose score:", round(float(pred_score), 2))
print("Reference dose score:", round(float(gt_score), 2))
print()
print(pred_score_df[["Structure", "MetricType", "MetricValue", "Score", "Status"]]
      .round(3).to_string(index=False))


## 10. 可视化

DosePy 提供两类图：

1. **不依赖 PortPy Plan 的图**（推荐用于预测 vs GT）：
   - `plot_slice_comparison`：CT + 预测 + GT + 差分；
   - `plot_dvh_comparison`：多曲线 DVH；
   - `plot_structure_metric_bars`：分结构指标；
   - `save_evaluation_report`：一键输出 PNG + CSV + summary。
2. **PortPy plan 级图**（需要 `my_plan`）：
   - `plot_plan_dvh`；
   - `plot_plan_2d_slice`；
   - `plot_plan_fluence_3d`。

### 10.1 切片对比

默认选 PTV 最大的那一层。也可以手动传 `slice_idx=60`。


In [ ]:
plot_slice_comparison(
    ct_hu,
    {"Predicted": pred_dose_3d, "Reference": gt_dose_3d},
    structures=structure_masks,
    prescription_gy=RX,
)
plt.show()


### 10.2 DVH 对比

同一结构用同一颜色，不同剂量曲线用不同线型。预测（实线）和 GT（虚线）的差距越小越好。


In [ ]:
plot_dvh_comparison(
    {"Predicted": pred_dose_3d, "Reference": gt_dose_3d},
    structure_masks,
    prescription_gy=RX,
    title="{}: predicted vs reference".format(PATIENT_ID),
)
plt.show()


### 10.3 分结构 MAE



In [ ]:
plot_structure_metric_bars(per_structure, metric="MAE")
plt.show()


### 10.4 一键生成评估报告

`save_evaluation_report` 会在 `REPORT_DIR` 下生成：

- `<Patient>_metrics.csv`
- `<Patient>_dvh_metrics.csv`
- `<Patient>_slices.png`
- `<Patient>_dvh.png`
- `<Patient>_summary.txt`
- 如果传入 scorecard，还会生成 `<Patient>_scorecard.csv`


In [ ]:
paths = save_evaluation_report(
    out_dir=REPORT_DIR,
    patient_id=PATIENT_ID,
    ct_hu=ct_hu,
    doses={"Predicted": pred_dose_3d, "Reference": gt_dose_3d},
    structures=structure_masks,
    prescription_gy=RX,
    metrics_df=per_structure,
    dvh_df=dvh_df,
    score=pred_score,
    score_df=pred_score_df,
)
for key, value in paths.items():
    print(f"{key:>15}: {value}")


### 10.5 PortPy plan 级可视化（可选）

这些函数直接复用 PortPy 的 `Visualization`，需要 `my_plan` 和 1D 剂量。适合做最终汇报图。


In [ ]:
# 2D 切片：预测剂量
SLICE_NUM = ct_hu.shape[0] // 2
plot_plan_2d_slice(
    my_plan,
    dose_1d=pred_dose_1d,
    slice_num=SLICE_NUM,
    struct_names=[name_map[s] for s in ["PTV", "CORD", "ESOPHAGUS", "HEART"] if s in name_map],
    show_dose=True,
)
plt.show()

# DVH：预测（实线） vs 参考（虚线）
fig, ax = plt.subplots(figsize=(10, 6))
plot_plan_dvh(
    my_plan,
    dose_1d=gt_dose_1d,
    struct_names=[name_map[s] for s in ["PTV", "CORD", "ESOPHAGUS", "HEART"] if s in name_map],
    style="dotted",
    ax=ax,
)
plot_plan_dvh(
    my_plan,
    dose_1d=pred_dose_1d,
    struct_names=[name_map[s] for s in ["PTV", "CORD", "ESOPHAGUS", "HEART"] if s in name_map],
    style="solid",
    ax=ax,
)
ax.set_title("PortPy DVH: predicted (solid) vs reference (dotted)")
plt.show()


## 11. 下游：用预测剂量做通量优化

预测剂量本身不是可交付计划。PortPy 的做法是用预测剂量作为目标，求解 beamlet fluence：

$$
\min_x \; \|A^{PTV}x - pred^{PTV}/n_{fx}\|^2
+ w_{OAR}\|\max(A^{OAR}x - pred^{OAR}/n_{fx}, 0)\|^2
+ w_s \cdot \text{smoothness}(x)
\quad \text{s.t.}\quad x \ge 0
$$

- `A`：influence matrix；
- `x`：beamlet intensity；
- `pred`：本教程得到的预测总剂量；
- `n_fx`：分次数；
- 平滑项目的是让 fluence 更可交付。

求解后得到的 `dose_1d` 是**优化后的总剂量**，可以再算 DVH 和 scorecard。


In [ ]:
optimized = optimize_fluence_to_prediction(
    my_plan,
    pred_dose_1d,
    num_fractions=clinical_criteria.get_num_of_fractions(),
    ptv_weight=1.0,
    oar_weight=1.0,
    smoothness_weights=(0.6, 0.4),
    solver="MOSEK",       # 没有 MOSEK 可改 "SCS" 或 "CLARABEL"
    verbose=False,
)

opt_dose_1d = optimized["dose_1d"]
opt_dose_3d = inf_matrix.dose_1d_to_3d(dose_1d=opt_dose_1d)

print("optimized dose range: [{:.2f}, {:.2f}] Gy".format(
    float(opt_dose_3d.min()), float(opt_dose_3d.max())
))


### 11.1 优化后剂量 vs 预测 vs 参考

注意：优化后的剂量是在**真实 influence matrix** 上算出来的；预测剂量只是网络输出。比较三者可以看出“预测误差”和“优化可交付性”各占多少。


In [ ]:
plot_dvh_comparison(
    {
        "Predicted": pred_dose_3d,
        "Reference": gt_dose_3d,
        "Optimized": opt_dose_3d,
    },
    structure_masks,
    prescription_gy=RX,
    title="{}: predicted / reference / optimized".format(PATIENT_ID),
)
plt.show()


In [ ]:
opt_score, opt_score_df = get_plan_score(
    my_plan,
    opt_dose_1d,
    scorecard,
    prescription_gy=RX,
    alias_map=alias_map,
    return_score_df=True,
)

print("Predicted dose score :", round(float(pred_score), 2))
print("Reference dose score :", round(float(gt_score), 2))
print("Optimized plan score :", round(float(opt_score), 2))


## 12. 自定义与扩展

### 12.1 新增 site

在 `dosepy/preprocess/site_config.py` 的 `SITE_PROFILES` 中新增一个 `SiteProfile`：

```python
"headneck": SiteProfile(
    site="headneck",
    disease_struct_names=("parotid_l", "parotid_r", "cord", "ptv"),
    labels={"parotid_l": 1, "parotid_r": 2, "cord": 3, "ptv": 1},
    prescription_gy=70.0,
    oar_channels=(("PAROTID_L", 1), ("PAROTID_R", 2), ("CORD", 3)),
    dvh_structures=("PTV", "PAROTID_L", "PAROTID_R", "CORD"),
    default_protocol="HeadNeck_...",
),
```

然后把 `data_preprocess(site="headneck")`、`train`、`test`、评估都切到该 site。注意 7 通道 = CT + 4 个 OAR + 1 个空 + PTV；如果 OAR 多于 4 个，需要改 `transform_3d_data` 的 `one_hot(..., 6)` 和 `input_nc`，并重新训练。BEAM 已移除，不要按 8 通道布局训练。

### 12.2 新增模型

1. 在 `dosepy/models/` 下新建 `<name>_model.py`，定义 `<Name>Model(BaseModel)`；
2. 在 `networks3d.define_G` 中注册新的 `netG`；
3. 训练时设 `netG="<name>"`。
4. 新模型必须实现 `set_input`、`forward`、`optimize_parameters`、`calculate_validation_loss`；建议实现 `calculate_validation_metrics`。

### 12.3 新增损失

在 `dosepy/models/losses3d.py` 中实现 `nn.Module`，然后在 `networks3d.py` 的 re-export 和 `DosePrediction3DModel.backward_G` 中接入，最后在 `train_options.py` 的 `--dose_loss` choices 中增加选项。

### 12.4 数据增强

`train_options["augment"] = True` 会启用 `base_dataset.augment_sample`：随机翻转、±10° 旋转、±10 体素平移，CT/dose 线性插值，mask 最近邻。注意 dose 在变换后会重新乘回 dose>0 的 mask，避免空气区域出现假剂量。

### 12.5 显存不足

- 把 `batch_size` 降到 1；
- 用 `unet_128` 而不是 `unet_256` / `mednext`；
- 关闭 `augment`（增强本身不占太多显存，但会拖慢）；
- 用 `torch.cuda.empty_cache()`（`train.py` 开头已经调用一次）；
- 用混合精度需要自己改 `train.py`，当前版本没有内置 AMP。

### 12.6 加载 DosePy 7 通道 checkpoint

本版本是 anatomy-only 7 通道，**不能加载 PortPy 官方 8 通道 checkpoint**（第一层卷积的输入通道数不同，会报 size mismatch）。加载 DosePy 自己训练的 7 通道 checkpoint 时：

- 保持 7 通道顺序 `[CT, OAR..., PTV]`；
- 保持 CT 归一化；
- `netG`、`ngf`、`mednext_model_id`、`mednext_kernel_size`、`mednext_n_channels` 与训练时一致；
- checkpoint 命名 `<epoch>_net_G.pth`。

```python
test_options["epoch"] = "best_mae"
test_options["netG"] = NET_G       # 与训练一致
```

如果要重新引入 BEAM 通道，需要同时把 `input_nc` 改回 8、恢复 dataset 的 BEAM 拼接与预处理，并重新训练。


## 13. 常见问题（FAQ）

### Q1: `ModuleNotFoundError: No module named 'nnunet_mednext'`

说明选了 `mednext*` 但没装 MedNeXt。安装：

```bash
pip install git+https://github.com/MIC-DKFZ/MedNeXt.git
```

或者改用 `NET_G = "unet_128"`。

### Q2: `RuntimeError: gpu_ids=[0] was requested but CUDA is not available`

把 `gpu_ids` 设为 `[]`（notebook）或 `-1`（CLI）。

### Q3: `ValueError: Unknown option(s) [...]`

`TrainOptions` / `TestOptions` 中没有这个参数，通常是拼写错误。报错信息会列出所有可用 key。

### Q4: `FileNotFoundError: .../<Patient>_CT2DOSE.nrrd`

检查：

- `model_name` / `EXPERIMENT` 是否和训练时一致；
- `epoch` 是否有对应 checkpoint（`best_mae` 可能没生成，改用 `latest`）；
- `checkpoints_dir` / `results_dir` 是否是绝对路径；
- 推理是否真的写入了 `results/<name>/test_<epoch>/npz_images/`。

### Q5: `KeyError: 'DOSE'`

用 `SingleDataset`（推理）读取了训练 npz，或者用 `DosePred3DDataset`（测试）读取了没有 GT 的推理 npz。检查 `model` / `dataset_mode` 组合：

- 有 GT：`model="doseprediction3d"`，`dataset_mode="dosepred3d"`；
- 无 GT：`model="test"`，`dataset_mode="single"`。

### Q6: `MOSEK solver is not installed`

下游通量优化默认 MOSEK。没有 License 时：

```python
optimize_fluence_to_prediction(..., solver="SCS")
# 或 "CLARABEL"
```

### Q7: checkpoint 加载时报 `size mismatch`

网络结构或超参与训练时不一致。重点检查：

- `netG`；
- `ngf`（U-Net）；
- `mednext_model_id` / `mednext_kernel_size` / `mednext_n_channels`；
- `input_nc` / `output_nc`。

### Q8: 预处理时提示 `Dose file does not exist`

`rt_dose_echo_<technique>.dcm` 文件名不匹配。确认 `TECHNIQUE` 与实际文件一致（例如 `imrt`、`vmat`），或检查 `DicomFiles/` 内容。

### Q9: 训练一开始 val MAE 就是 `nan`

- 检查 `val` 目录是否为空；
- 检查 `PRESCRIPTION_GY` 是否存在（challenge-style MAE 需要它；没有时会 fallback 到 1.0，但不推荐）；
- 检查 `BODY` 是否存在（`mednext_hybrid` 需要）。

### Q10: Jupyter 里 `parse()` 报 unrecognized arguments

DosePy 已经通过 `parse(argv=[], overrides=...)` / `build_opt` 解决。不要直接调用 `TrainOptions().parse()` 而不传 `argv`，除非你确实想解析 `sys.argv`。

### Q11: 为什么没有 BEAM 通道？能加载 PortPy 官方 checkpoint 吗？

本版本按 anatomy-only 设计：输入只有 `CT + OAR one-hot + PTV`，共 7 通道，没有 BEAM。PortPy 官方 total-dose checkpoint 是第一层 8 通道（含 BEAM），因此**不能加载**，会报 `size mismatch`；需要按本 notebook 重新训练 7 通道模型。

如果确实需要恢复 BEAM，需要同时修改：

- `data_preprocess.py` / `predict.py`：重新生成 BEAM；
- `base_dataset.py` / `dosepred3d_dataset.py` / `single_dataset.py`：重新拼接 BEAM；
- `input_nc` 改回 8；
- 重新训练。

只改 `input_nc` 或只改 README 都会导致维度不匹配。


## 14. 端到端最小脚本（复制到 `.py` 也可运行）

把下面的常量改成你的路径，就可以不用 notebook 跑完整流程。建议先用 `n_epochs=2` 做 smoke test。

```python
from dosepy.preprocess import data_preprocess, split_preprocessed_dataset, predict_using_model
from dosepy.train import train
from dosepy.test import test
from dosepy.evaluation import dose_metrics, dose_metrics_by_structure, dvh_table
from dosepy.visualization import plot_slice_comparison, plot_dvh_comparison

DATA_DIR = "/path/to/PortPy/data"
OUT_DIR = "/path/to/work/preprocessed_lung"
CKPT_DIR = "/path/to/work/checkpoints"
RESULT_DIR = "/path/to/work/results"
SITE, PROTOCOL, TECH = "lung", "Lung_2Gy_30Fx", "imrt"
NAME = "dosepy_lung_unet128"

data_preprocess(DATA_DIR, OUT_DIR, site=SITE, protocol_name=PROTOCOL, technique_name=TECH)
split_preprocessed_dataset(OUT_DIR, site=SITE)

history = train({
    "dataroot": OUT_DIR, "checkpoints_dir": CKPT_DIR, "name": NAME,
    "model": "doseprediction3d", "dataset_mode": "dosepred3d",
    "netG": "unet_128", "dose_loss": "mae",
    "input_nc": 7, "output_nc": 1, "batch_size": 1,
    "n_epochs": 2, "gpu_ids": [0],
})

test({
    "dataroot": OUT_DIR, "checkpoints_dir": CKPT_DIR, "results_dir": RESULT_DIR,
    "name": NAME, "model": "doseprediction3d", "dataset_mode": "dosepred3d",
    "netG": "unet_128", "dose_loss": "mae",
    "input_nc": 7, "output_nc": 1, "epoch": "best_mae", "eval": True,
    "gpu_ids": [0],
})

pred = predict_using_model(
    "Lung_Patient_94", DATA_DIR, out_dir="/path/to/work/infer",
    model_name=NAME, checkpoints_dir=CKPT_DIR, results_dir=RESULT_DIR,
    netG="unet_128", site=SITE, protocol_name=PROTOCOL, epoch="best_mae",
)
```

---

### 完成检查清单

- [ ] `DATA_DIR` 指向正确的 PortPy 数据根目录
- [ ] `data_preprocess` 输出 npz，且形状为 `128³`
- [ ] `split_preprocessed_dataset` 生成 train/val/test
- [ ] `train` 完成，存在 `best_mae_net_G.pth`
- [ ] `test` 生成 `patient_metrics.csv`
- [ ] `predict_using_model` 返回 `(D,H,W)` 的 CT 网格剂量
- [ ] 评估指标、DVH、scorecard 均能生成
- [ ] 切片图和 DVH 图符合预期
- [ ] （可选）下游通量优化成功求解

如果以上都通过，DosePy 的全剂量预测流程就完整跑通了。
